# QuantUI-local — Quantum Chemistry Calculator

Run PySCF quantum chemistry calculations directly in this notebook.

**How to use:**
1. Select or enter a molecule in **Molecule Input**
2. Choose a method and basis set in **Calculation Setup**
3. Click **Run Calculation** — results appear below
4. Optionally add results to **Compare** or **Export** a standalone script

> **Platform note:** PySCF requires Linux, macOS, or WSL.  \
> Windows users: `apptainer run quantui-local.sif`


In [ ]:
# Environment check — verifies correct conda environment.
# Tagged skip-execution and remove-input so it is hidden in Voilà.
import sys as _sys
_env = _sys.prefix
if "quantui" not in _env.lower():
    print(f"Warning: active environment may not be quantui-local")
    print(f"Active: {_env}")
    print("Run: conda activate quantui-local")


In [ ]:
import threading
import ipywidgets as widgets
from IPython.display import display, HTML

import quantui
from quantui import (
    Molecule, parse_xyz_input,
    MOLECULE_LIBRARY, SUPPORTED_METHODS, SUPPORTED_BASIS_SETS,
    DEFAULT_METHOD, DEFAULT_BASIS, DEFAULT_CHARGE, DEFAULT_MULTIPLICITY,
    session_can_handle, get_session_resources,
    PUBCHEM_AVAILABLE, VISUALIZATION_AVAILABLE, ASE_AVAILABLE,
    QUICK_START_TEMPLATES,
)

# Optional — degrade gracefully if unavailable
try:
    from quantui import run_in_session, SessionResult
    PYSCF_AVAILABLE = True
except (ImportError, AttributeError):
    PYSCF_AVAILABLE = False

try:
    from quantui import student_friendly_fetch
except (ImportError, AttributeError):
    student_friendly_fetch = None

try:
    from quantui import display_molecule
except (ImportError, AttributeError):
    display_molecule = None

try:
    from quantui import preoptimize
    PREOPT_AVAILABLE = True
except (ImportError, AttributeError):
    PREOPT_AVAILABLE = False

# Mutable session state shared across all callbacks
_state = {"molecule": None, "last_result": None, "results": []}

# ── Dark mode toggle ─────────────────────────────────────────────────────────
# Uses CSS filter invert+hue-rotate on the html element so it works with all
# inline-styled elements. canvas/img/iframe are re-inverted to keep their
# original appearance (e.g. the py3Dmol 3D viewer).
_DARK_CSS = (
    "<style>"
    "html { filter: invert(1) hue-rotate(180deg) !important; }"
    "canvas, img, iframe, video "
    "{ filter: invert(1) hue-rotate(180deg) !important; }"
    "</style>"
)

_theme_style = widgets.Output(
    layout=widgets.Layout(height="0px", overflow="hidden", margin="0", padding="0")
)

dark_mode_btn = widgets.ToggleButton(
    value=False,
    description=" Dark mode",
    icon="moon-o",
    tooltip="Toggle dark / light mode",
    layout=widgets.Layout(width="136px", height="30px"),
)


def _toggle_theme(change):
    _theme_style.clear_output()
    if change["new"]:
        dark_mode_btn.icon = "sun-o"
        dark_mode_btn.description = " Light mode"
        with _theme_style:
            display(HTML(_DARK_CSS))
    else:
        dark_mode_btn.icon = "moon-o"
        dark_mode_btn.description = " Dark mode"


dark_mode_btn.observe(_toggle_theme, names="value")

display(widgets.HBox(
    [dark_mode_btn],
    layout=widgets.Layout(justify_content="flex-end", margin="0 0 4px"),
))
display(_theme_style)

# ── Status panel ────────────────────────────────────────────────────────────
_cores, _mem_gb = get_session_resources()
_mem = f"{_mem_gb} GB" if _mem_gb is not None else "unknown"


def _ok(flag, extra=""):
    tick = '<span style="color:green">&#10003;</span>'
    cross = '<span style="color:#c00">&#10007;</span>'
    return (tick if flag else cross) + (" " + extra if extra else "")


_items = [
    ("PySCF (calculations)", _ok(PYSCF_AVAILABLE,
        "" if PYSCF_AVAILABLE else "&mdash; Linux / macOS / WSL required")),
    ("ASE (structure I/O, opt.)", _ok(ASE_AVAILABLE)),
    ("PubChem search", _ok(PUBCHEM_AVAILABLE)),
    ("3D viewer (py3Dmol)", _ok(VISUALIZATION_AVAILABLE)),
    ("CPU cores / Memory", f"<b>{_cores}</b> cores / <b>{_mem}</b>"),
]
_rows = "".join(
    f'<tr><td style="padding:2px 14px 2px 0;color:#555">{k}</td><td>{v}</td></tr>'
    for k, v in _items
)
display(HTML(
    f'<div style="background:#f5f5f5;border-left:4px solid #2196F3;'
    f'padding:10px 14px;border-radius:4px;margin:8px 0">'
    f"<b>QuantUI-local {quantui.__version__}</b>"
    f'<table style="margin-top:6px;font-size:13px">{_rows}</table>'
    f"</div>"
))


In [ ]:
# ── Shared output widgets ────────────────────────────────────────────────────
mol_info_html = widgets.HTML(
    value='<i style="color:#888">No molecule loaded yet.</i>'
)
viz_output    = widgets.Output(layout=widgets.Layout(min_height="50px"))
run_output    = widgets.Output(
    layout=widgets.Layout(
        border="1px solid #c0ccd8", min_height="80px", max_height="400px",
        padding="8px", overflow_y="auto",
    )
)
with run_output:
    display(HTML(
        '<p style="color:#999;font-style:italic;font-size:13px;margin:2px 0">'
        "No calculation run yet. PySCF output and any errors will appear here."
        "</p>"
    ))
result_output     = widgets.Output()
comparison_output = widgets.Output()
notes_output      = widgets.Output()

# ── Calculation setup (defined here so _set_molecule can update them) ────────
method_dd = widgets.Dropdown(
    options=SUPPORTED_METHODS, value=DEFAULT_METHOD,
    description="Method:", style={"description_width": "100px"},
    layout=widgets.Layout(width="260px"),
)
basis_dd = widgets.Dropdown(
    options=SUPPORTED_BASIS_SETS, value=DEFAULT_BASIS,
    description="Basis Set:", style={"description_width": "100px"},
    layout=widgets.Layout(width="260px"),
)
charge_si = widgets.BoundedIntText(
    value=DEFAULT_CHARGE, min=-10, max=10,
    description="Charge:", style={"description_width": "100px"},
    layout=widgets.Layout(width="190px"),
)
mult_si = widgets.BoundedIntText(
    value=DEFAULT_MULTIPLICITY, min=1, max=10,
    description="Multiplicity:", style={"description_width": "100px"},
    layout=widgets.Layout(width="190px"),
)
preopt_cb = widgets.Checkbox(
    value=False,
    description="Pre-optimize geometry (fast LJ force-field)",
    disabled=not PREOPT_AVAILABLE,
    layout=widgets.Layout(width="400px"),
)

# ── Run widgets ──────────────────────────────────────────────────────────────
run_btn = widgets.Button(
    description="Run Calculation", button_style="success", icon="play",
    disabled=True, layout=widgets.Layout(width="200px", height="36px"),
)
run_status = widgets.Label()

# ── Log clear button ─────────────────────────────────────────────────────────
log_clear_btn = widgets.Button(
    description="Clear", button_style="", icon="times",
    layout=widgets.Layout(width="90px", height="26px"),
    tooltip="Clear calculation output",
)

def _clear_log(btn):
    run_output.clear_output()

log_clear_btn.on_click(_clear_log)

# ── Comparison / export widgets ───────────────────────────────────────────────
accumulate_btn = widgets.Button(
    description="Add to Comparison", button_style="info", icon="plus",
    disabled=True, layout=widgets.Layout(width="190px"),
)
clear_btn = widgets.Button(
    description="Clear", button_style="warning", icon="trash",
    layout=widgets.Layout(width="100px"),
)
export_btn = widgets.Button(
    description="Export Script", button_style="", icon="download",
    disabled=True, layout=widgets.Layout(width="160px"),
)
export_status = widgets.Label()


# ── Callbacks ─────────────────────────────────────────────────────────────────

def _set_molecule(mol, label=""):
    # Update shared state and refresh dependent widgets.
    _state["molecule"] = mol
    run_btn.disabled    = False
    export_btn.disabled = False

    try:
        n_e = mol.get_electron_count()
        e_str = f"{n_e} electrons"
    except Exception:
        e_str = ""

    _lbl = f'<br><small style="color:#777">{label}</small>' if label else ""
    mol_info_html.value = (
        f'<b style="font-size:15px">{mol.get_formula()}</b>'
        f'&ensp;<span style="color:#555;font-size:13px">'
        f"{len(mol.atoms)} atoms"
        + (f" &bull; {e_str}" if e_str else "")
        + f" &bull; charge {mol.charge} &bull; mult {mol.multiplicity}"
        + f"</span>{_lbl}"
    )
    charge_si.value = mol.charge
    mult_si.value   = mol.multiplicity
    if mol.multiplicity > 1 and method_dd.value == "RHF":
        method_dd.value = "UHF"

    viz_output.clear_output(wait=True)
    if display_molecule is not None:
        with viz_output:
            display_molecule(mol)

    _update_notes()


def _format_result(r):
    _conv   = "Yes" if r.converged else "No (treat results with caution)"
    _cc     = "green" if r.converged else "#c00"
    _gap    = (
        f"{r.homo_lumo_gap_ev:.4f} eV"
        if r.homo_lumo_gap_ev is not None else "N/A"
    )
    _rows = "".join(
        f'<tr>'
        f'<td style="padding:3px 18px 3px 0;color:#444">{k}</td>'
        f'<td style="color:{vc}">{v}</td>'
        f"</tr>"
        for k, v, vc in [
            ("Total energy",
             f"{r.energy_hartree:.8f} Ha &ensp;({r.energy_ev:.4f} eV)", "#000"),
            ("HOMO-LUMO gap",   _gap, "#000"),
            ("SCF converged",   _conv, _cc),
            ("SCF iterations",  str(r.n_iterations), "#000"),
        ]
    )
    return (
        f'<div style="background:#f0fff0;border-left:4px solid #4CAF50;'
        f'padding:10px 14px;border-radius:4px;margin:6px 0">'
        f"<b>{r.formula} &mdash; {r.method}/{r.basis}</b>"
        f'<table style="margin-top:8px;font-size:14px;border-collapse:collapse">'
        f"{_rows}</table></div>"
    )


def _do_run(btn):
    mol = _state["molecule"]
    if mol is None:
        run_status.value = "Load a molecule first."
        return
    run_btn.disabled = True
    run_status.value = "Starting..."
    run_output.clear_output()
    result_output.clear_output()

    def _thread():
        try:
            calc_mol = mol
            if preopt_cb.value and PREOPT_AVAILABLE:
                run_status.value = "Pre-optimizing..."
                # preoptimize returns (molecule, rmsd) — unpack both
                calc_mol, _rmsd = preoptimize(mol)
                _set_molecule(calc_mol, f"Geometry pre-optimized (LJ force-field, RMSD={_rmsd:.3f} Å)")

            run_status.value = "Calculating..."
            with run_output:
                result = run_in_session(
                    molecule=calc_mol,
                    method=method_dd.value,
                    basis=basis_dd.value,
                )

            _state["last_result"] = result
            accumulate_btn.disabled = False

            # append_display_data is thread-safe; display() inside with-block is not
            result_output.append_display_data(HTML(_format_result(result)))

            run_status.value = "Done."

        except ImportError:
            # Only catch ImportError for PySCF-unavailable message — not all errors
            run_output.append_stdout(
                "PySCF is not available in this environment.\n"
                "PySCF requires Linux, macOS, or WSL.\n"
                "On Windows: use the Apptainer container.\n"
                "  apptainer run quantui-local.sif\n"
            )
            run_status.value = "PySCF unavailable."

        except Exception as exc:
            import traceback
            run_output.append_stdout(
                f"Error: {exc}\n\n{traceback.format_exc()}"
            )
            run_status.value = "Error — see Calculation Output below."

        finally:
            run_btn.disabled = False

    threading.Thread(target=_thread, daemon=True).start()

run_btn.on_click(_do_run)


def _update_notes(change=None):
    notes_output.clear_output(wait=True)
    mol = _state["molecule"]
    if mol is None:
        return
    try:
        from quantui import PySCFCalculation
        calc  = PySCFCalculation(mol, method=method_dd.value, basis=basis_dd.value)
        notes = calc.get_educational_notes()
        if notes:
            safe = (
                notes
                .replace("**", "<b>", 1)
                .replace("**", "</b>", 1)
                .replace("\n\n", "<br><br>")
            )
            with notes_output:
                display(HTML(
                    '<div style="background:#fffbf0;padding:8px 12px;'
                    'border-radius:4px;font-size:13px;margin-top:6px">'
                    + safe + "</div>"
                ))
    except Exception:
        pass

method_dd.observe(_update_notes, names="value")
basis_dd.observe(_update_notes, names="value")


def _do_accumulate(btn):
    r = _state["last_result"]
    if r is None:
        return
    _state["results"].append(r)
    _refresh_comparison()

accumulate_btn.on_click(_do_accumulate)


def _refresh_comparison():
    from quantui import summary_from_session_result, comparison_table_html
    comparison_output.clear_output(wait=True)
    results = _state["results"]
    if not results:
        return
    summaries = [summary_from_session_result(r) for r in results]
    with comparison_output:
        display(HTML(comparison_table_html(summaries)))
        if len(summaries) > 1:
            try:
                from quantui import plot_comparison
                plot_comparison(summaries)
            except Exception:
                pass


def _do_clear(btn):
    _state["results"].clear()
    comparison_output.clear_output()

clear_btn.on_click(_do_clear)


def _do_export(btn):
    mol = _state["molecule"]
    if mol is None:
        export_status.value = "Load a molecule first."
        return
    try:
        from quantui import PySCFCalculation
        from pathlib import Path
        calc  = PySCFCalculation(mol, method=method_dd.value, basis=basis_dd.value)
        fname = f"{mol.get_formula()}_{method_dd.value}_{basis_dd.value}.py"
        calc.generate_calculation_script(Path(fname))
        export_status.value = f"Saved: {fname}"
    except Exception as exc:
        export_status.value = f"Error: {exc}"

export_btn.on_click(_do_export)


In [ ]:
# ── Preset Library ─────────────────────────────────────────────────────────
_preset_opts = ["(select a molecule)"] + list(MOLECULE_LIBRARY.keys())
preset_dd = widgets.Dropdown(
    options=_preset_opts, value="(select a molecule)",
    description="Molecule:", style={"description_width": "90px"},
    layout=widgets.Layout(width="320px"),
)

def _load_preset(change):
    name = change["new"]
    if name.startswith("("):
        return
    d = MOLECULE_LIBRARY[name]
    _set_molecule(
        Molecule(
            atoms=d["atoms"], coordinates=d["coordinates"],
            charge=d["charge"], multiplicity=d["multiplicity"],
        ),
        d["description"],
    )

preset_dd.observe(_load_preset, names="value")

# ── XYZ Input ──────────────────────────────────────────────────────────────
xyz_area = widgets.Textarea(
    placeholder=(
        "Paste XYZ coordinates (symbol  x  y  z):\n"
        "O  0.000  0.000  0.000\n"
        "H  0.757  0.587  0.000\n"
        "H -0.757  0.587  0.000"
    ),
    layout=widgets.Layout(width="440px", height="130px"),
)
xyz_btn = widgets.Button(description="Load XYZ", button_style="info", icon="upload")
xyz_msg = widgets.Label()

def _load_xyz(btn):
    try:
        mol = parse_xyz_input(xyz_area.value.strip())
        _set_molecule(mol, "Loaded from XYZ input")
        xyz_msg.value = ""
    except Exception as exc:
        xyz_msg.value = f"Parse error: {exc}"

xyz_btn.on_click(_load_xyz)

# ── PubChem Search ─────────────────────────────────────────────────────────
pubchem_txt = widgets.Text(
    placeholder="name or SMILES  (e.g. aspirin, caffeine, CC(=O)O)",
    layout=widgets.Layout(width="380px"),
)
pubchem_btn = widgets.Button(
    description="Search", button_style="info", icon="search",
    disabled=not PUBCHEM_AVAILABLE,
    layout=widgets.Layout(width="100px"),
)
pubchem_msg = widgets.Label(
    value="" if PUBCHEM_AVAILABLE else "PubChem unavailable — check internet connection"
)

def _search_pubchem(btn):
    query = pubchem_txt.value.strip()
    if not query:
        pubchem_msg.value = "Enter a molecule name or SMILES."
        return
    if student_friendly_fetch is None:
        pubchem_msg.value = "PubChem module not available."
        return
    pubchem_msg.value = f'Searching for "{query}"...'
    pubchem_btn.disabled = True

    def _do():
        try:
            mol = student_friendly_fetch(query)
            _set_molecule(mol, f"PubChem: {query}")
            pubchem_msg.value = f"Loaded {mol.get_formula()} from PubChem."
        except Exception as exc:
            pubchem_msg.value = f"Not found: {exc}"
        finally:
            pubchem_btn.disabled = False

    threading.Thread(target=_do, daemon=True).start()

pubchem_btn.on_click(_search_pubchem)

# ── Assemble and display ───────────────────────────────────────────────────
_hint = '<p style="margin:4px 0 8px;color:#666;font-size:13px">'
tab_preset = widgets.VBox([
    widgets.HTML(_hint + "Choose from 20+ curated educational molecules.</p>"),
    preset_dd,
])
tab_xyz = widgets.VBox([
    widgets.HTML(_hint + "Paste XYZ coordinates (element x y z, one atom per line).</p>"),
    xyz_area,
    widgets.HBox([xyz_btn, xyz_msg]),
])
tab_pubchem = widgets.VBox([
    widgets.HTML(
        _hint + "Search by name or SMILES. Requires internet connection.</p>"
    ),
    widgets.HBox([pubchem_txt, pubchem_btn]),
    pubchem_msg,
])

input_tab = widgets.Tab(children=[tab_preset, tab_xyz, tab_pubchem])
for _i, _t in enumerate(["Preset Library", "XYZ Input", "PubChem Search"]):
    input_tab.set_title(_i, _t)

display(HTML(
    '<h3 style="margin:8px 0 6px;border-bottom:1px solid #ddd;padding-bottom:4px">'
    "Molecule Input</h3>"
))
display(input_tab)
display(widgets.HTML('<div style="height:8px"></div>'))
display(mol_info_html)
display(viz_output)


In [ ]:
display(HTML(
    '<h3 style="margin:14px 0 6px;border-bottom:1px solid #ddd;padding-bottom:4px">'
    "Calculation Setup</h3>"
))
display(widgets.HBox([
    widgets.VBox([method_dd, basis_dd]),
    widgets.HTML("&ensp;&ensp;"),
    widgets.VBox([charge_si, mult_si]),
]))
display(preopt_cb)
display(notes_output)


In [ ]:
display(HTML(
    '<h3 style="margin:14px 0 6px;border-bottom:1px solid #ddd;padding-bottom:4px">'
    "Run Calculation</h3>"
    '<p style="color:#555;font-size:13px;margin:0 0 8px">PySCF runs in this kernel. '
    "Output appears live below. Large molecules or high-accuracy basis sets may take "
    "several minutes on a laptop.</p>"
))
display(widgets.HBox([run_btn, run_status]))
display(widgets.HBox(
    [
        widgets.HTML(
            '<span style="font-size:13px;font-weight:600;color:#444">'
            "Calculation Output</span>"
        ),
        log_clear_btn,
    ],
    layout=widgets.Layout(align_items="center", justify_content="space-between",
                          margin="10px 0 4px", max_width="460px"),
))
display(run_output)


In [ ]:
display(HTML(
    '<h3 style="margin:14px 0 6px;border-bottom:1px solid #ddd;padding-bottom:4px">'
    "Results</h3>"
))
display(result_output)


In [ ]:
display(HTML(
    '<h3 style="margin:14px 0 6px;border-bottom:1px solid #ddd;padding-bottom:4px">'
    "Compare Results</h3>"
    '<p style="color:#555;font-size:13px;margin:0 0 8px">'
    "Run multiple calculations then add each one to compare energies side-by-side.</p>"
))
display(widgets.HBox([accumulate_btn, clear_btn]))
display(comparison_output)

display(HTML(
    '<h3 style="margin:14px 0 6px;border-bottom:1px solid #ddd;padding-bottom:4px">'
    "Export Standalone Script</h3>"
    '<p style="color:#555;font-size:13px;margin:0 0 8px">'
    "Download a self-contained PySCF script you can study or run outside the notebook.</p>"
))
display(widgets.HBox([export_btn, export_status]))
